In [1]:
!pip install hexaly -i https://pip.hexaly.com

Looking in indexes: https://pip.hexaly.com


In [2]:
!pip install osmnx gurobipy

In [3]:
import os

os.environ["HEXALY_LICENSE_FILE"] = "license.dat"

In [4]:
import os

os.environ["GRB_LICENSE_FILE"] = "gurobi.lic"

import gurobipy as gp

m = gp.Model()
print("Gurobi license found.")

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2476248
Academic license 2476248 - for non-commercial use only - registered to rs___@up.edu.ph
Gurobi license found.


In [5]:
import os
print(os.listdir("/content"))

['.config', 'graph_creation.py', 'technicians.xlsx', '__pycache__', 'license.dat', '.ipynb_checkpoints', 'output', 'gurobi.lic', 'solver.py', 'cache', 'solver_hexaly.py', 'instance_creation.py', 'farmers.xlsx', 'sample_data']


In [7]:
import sys
sys.path.append("/content")

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import networkx as nx

from instance_creation import instance_creation
from graph_creation import load_graph
from solver import solve_technician_routing
from solver_hexaly import solve_technician_routing_hexaly



def plot_routes(arcs, origin_of, farmers, office_node, positions):

    G = nx.DiGraph()
    G.add_nodes_from(positions)

    tech_origins = set(origin_of.values())
    farmer_set = set(farmers)

    node_colors = []
    for node in G.nodes():
        if node == office_node:
            node_colors.append("gold")
        elif node in tech_origins:
            node_colors.append("cornflowerblue")
        elif node in farmer_set:
            node_colors.append("lightgreen")
        else:
            node_colors.append("lightgray")

    palette = ["tab:red", "tab:blue", "tab:purple", "tab:orange", "tab:cyan"]
    route_colors = {t: palette[i % len(palette)] for i, t in enumerate(arcs)}

    for t, arc_list in arcs.items():
        color = route_colors[t]
        for a, b in arc_list:
            G.add_edge(a, b, color=color)

    edge_colors = [G[u][v]["color"] for u, v in G.edges()]

    node_labels = {}
    for node in G.nodes():
        if node == office_node:
            node_labels[node] = f"{node}\n(O)"
        elif node in tech_origins:
            tech_id = next(t for t, n in origin_of.items() if n == node)
            node_labels[node] = f"{node}\n({tech_id})"
        else:
            node_labels[node] = str(node)

    _, ax = plt.subplots(figsize=(9, 7))

    nx.draw_networkx_nodes(
        G, pos=positions, node_color=node_colors,
        node_size=900, ax=ax
    )

    nx.draw_networkx_labels(
        G, pos=positions, labels=node_labels,
        font_size=8, ax=ax
    )

    nx.draw_networkx_edges(
        G, pos=positions, edge_color=edge_colors,
        arrows=True, arrowsize=20,
        connectionstyle="arc3,rad=0.1", ax=ax
    )

    legend_handles = [
        mpatches.Patch(color="gold", label="Office"),
        mpatches.Patch(color="cornflowerblue", label="Technician home"),
        mpatches.Patch(color="lightgreen", label="Farmer"),
    ] + [
        Line2D([0], [0], color=c, linewidth=2, label=t)
        for t, c in route_colors.items()
    ]

    ax.legend(handles=legend_handles, loc="best")
    ax.set_title("Technician Routes")
    ax.axis("off")
    plt.tight_layout()
    plt.show()



if __name__ == "__main__":


    (all_nodes_ordered,
     farmer_nodes,
     technicians,
     office_node,
     initial_straws_new,
     distance_matrix,
     origin_of) = instance_creation()

    total_straws_available = 200

    hex_routes, hex_arcs, hex_RA = solve_technician_routing_hexaly(
        all_nodes=all_nodes_ordered,
        farmers=farmer_nodes,
        technicians=technicians,
        office_node=office_node,
        initial_straws=initial_straws_new,
        total_straws_available=total_straws_available,
        full_distance=distance_matrix,
        origin_of=origin_of,
        time_limit=100,
        verbose=True
    )

    print("\nHexaly done → starting Gurobi warm-start...\n")


    hexaly_x = {
        t: {i: {j: 0 for j in all_nodes_ordered} for i in all_nodes_ordered}
        for t in technicians
    }

    for t, arc_list in hex_arcs.items():
        for i, j in arc_list:
            hexaly_x[t][i][j] = 1

    hexaly_r = {
        t: {} for t in technicians
    }

    for t in technicians:
        for node in all_nodes_ordered:
            hexaly_r[t][node] = 0

    routes, arcs = solve_technician_routing(
        all_nodes=all_nodes_ordered,
        farmers=farmer_nodes,
        technicians=technicians,
        office_node=office_node,
        initial_straws=initial_straws_new,
        full_distance=distance_matrix,
        origin_of=origin_of,
        hexaly_x=hexaly_x,
        hexaly_r=hexaly_r,
        time_limit=500,
        verbose=True
    )


Loading graph from output/road_network.graphml ...
Graph loaded.
Nodes = 53804
Edges = 63450
True
Setting office at PCC-UPLB coordinates: (16.611823, 120.310104)
Nearest node to PCC-UPLB: 528712570
Selected 80 farmers, 24 technicians.
Loading distance matrix from output/full_distance_matrix.csv ...
Distance matrix loaded.
Sample distance: 
   tech0 -> farm0 = 52145.34 m
   Office -> tech0 = 77961.59 m
   Office -> farm0 = 105767.27 m

Model:
  expressions = 13352, operands = 166898
  decisions = 72 (bool = 0, int = 48, float = 0, interval = 0, optional interval = 0, list = 24, set = 0)
  constraints = 50, objectives = 1, constants = 9580

Param:
  time limit = 100 sec, no iteration limit, nb threads = 0
  nb displayed objectives = 7, nb displayed violated constraints = 6, warning level = 1
  seed = 0, annealing level = 1, time between displays = 1
  time between ticks = 1, iterations between ticks = 10000

[objective direction ]:     minimize

[  0 sec,       0 itr]: No feasible soluti

GurobiError: Unable to write to file 'logs/model.lp'